# 05 - Deploy Everywhere: De Modelo a App

Ya tenemos un modelo entrenado y exportado. Ahora vamos a desplegarlo
en las 4 plataformas principales.

## Contenido
1. Visión general del despliegue
2. Android: TFLite + CameraX
3. iOS: CoreML + Vision
4. Web: ONNX Runtime Web
5. Raspberry Pi + Coral Edge TPU
6. Scaffold automático con MLForge

## 1. Visión General

Cada plataforma tiene su runtime de inferencia óptimo:

```
                    ┌──→ Android (TFLite) ──→ .tflite
Modelo entrenado ──→├──→ iOS (CoreML)     ──→ .mlpackage
                    ├──→ Web (ONNX RT)    ──→ .onnx
                    └──→ RPi+Coral (TFLite) ──→ .tflite (int8)
```

### El pipeline completo:
```bash
# 1. Entrenar
mlforge train --config config.yaml

# 2. Exportar
mlforge export --config config.yaml --formats all

# 3. Generar proyecto de despliegue
mlforge deploy android --model exported/model.tflite --labels "rosa,tulipán,girasol"
```

## 2. Android: TFLite + CameraX

### Componentes:
- **TensorFlow Lite Interpreter**: ejecuta el modelo .tflite
- **CameraX**: API moderna de cámara de Android
- **ImageAnalysis**: procesa frames en tiempo real

### Código clave (Kotlin):

```kotlin
// Cargar modelo
val interpreter = Interpreter(loadModelFile("model.tflite"))

// Preprocesar imagen (resize + normalize)
val input = ByteBuffer.allocateDirect(1 * 224 * 224 * 3 * 4)
// ... fill buffer with normalized pixel values ...

// Inferencia
val output = Array(1) { FloatArray(NUM_CLASSES) }
interpreter.run(input, output)

// Obtener predicción
val maxIdx = output[0].indices.maxByOrNull { output[0][it] } ?: 0
val className = labels[maxIdx]
val confidence = output[0][maxIdx]
```

### Dependencias Gradle:
```kotlin
implementation("org.tensorflow:tensorflow-lite:2.14.0")
implementation("org.tensorflow:tensorflow-lite-support:0.4.4")
implementation("androidx.camera:camera-camera2:1.3.1")
```

MLForge genera un proyecto Android completo con `mlforge deploy android`.

## 3. iOS: CoreML + Vision

### Componentes:
- **CoreML**: framework de Apple para ML on-device
- **Vision**: procesa imágenes y conecta con CoreML
- **ANE (Apple Neural Engine)**: aceleración hardware en chips A/M

### Código clave (Swift):

```swift
// Cargar modelo
let model = try VNCoreMLModel(for: FlowerClassifier().model)

// Crear request
let request = VNCoreMLRequest(model: model) { request, error in
    guard let results = request.results as? [VNClassificationObservation] else { return }
    
    let topResult = results.first!
    print("\(topResult.identifier): \(topResult.confidence * 100)%")
}

// Ejecutar en imagen
let handler = VNImageRequestHandler(cgImage: image)
try handler.perform([request])
```

CoreML aprovecha automáticamente el Neural Engine de Apple (~15.8 TOPS en M1).

## 4. Web: ONNX Runtime Web

### Componentes:
- **ONNX Runtime Web**: ejecuta modelos ONNX en el navegador
- **WebGL/WebGPU backend**: aceleración GPU del navegador
- **Canvas API**: para captura de imagen y preprocesamiento

### Código clave (JavaScript):

```javascript
// Cargar modelo
const session = await ort.InferenceSession.create('model.onnx');

// Preprocesar imagen → tensor CHW normalizado
const tensor = new ort.Tensor('float32', preprocessed, [1, 3, 224, 224]);

// Inferencia
const results = await session.run({ image: tensor });
const predictions = results.predictions.data;

// Softmax + top-5
const probs = softmax(predictions);
const top5 = getTopK(probs, 5);
```

Ventajas de despliegue web:
- **Zero install**: funciona en cualquier navegador
- **Privacidad**: todo se ejecuta en el cliente, nada se sube al servidor
- **WebGPU**: nueva API con rendimiento cercano a nativo

## 5. Raspberry Pi + Google Coral Edge TPU

### Setup hardware:
```
RPi 4 (4GB) + Coral USB Accelerator + USB Camera
```

### Código clave (Python):

```python
from pycoral.adapters import classify, common
from pycoral.utils.edgetpu import make_interpreter

# Cargar modelo en el Edge TPU
interpreter = make_interpreter('model_edgetpu.tflite')
interpreter.allocate_tensors()

# Preprocesar imagen
size = common.input_size(interpreter)
image = Image.open('flower.jpg').resize(size)

# Inferencia (~3ms en Edge TPU!)
common.set_input(interpreter, image)
interpreter.invoke()
classes = classify.get_classes(interpreter, top_k=3)

for c in classes:
    print(f"{labels[c.id]}: {c.score:.2%}")
```

### Rendimiento típico:
| Modelo | CPU (RPi4) | Edge TPU |
|--------|-----------|----------|
| MobileNetV2 | ~130ms | ~8ms |
| EfficientNet-S | ~200ms | ~12ms |
| SSD MobileNet | ~250ms | ~15ms |

In [ ]:
# Demostración: inferencia con ONNX Runtime (funciona en cualquier máquina)
import onnxruntime as ort
import numpy as np
from PIL import Image
import torchvision.transforms as T

# Verificar que tenemos un modelo exportado
import os
onnx_path = 'output/model.onnx'

if not os.path.exists(onnx_path):
    print("Exporta un modelo primero:")
    print("  mlforge export --config config.yaml --formats onnx")
    print("\nO ejecuta el notebook 04 primero.")
else:
    # Cargar modelo
    session = ort.InferenceSession(onnx_path)
    input_name = session.get_inputs()[0].name
    
    # Crear input de ejemplo (imagen aleatoria para demo)
    dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)
    
    # Inferencia
    import time
    times = []
    for _ in range(50):
        start = time.perf_counter()
        result = session.run(None, {input_name: dummy})
        times.append((time.perf_counter() - start) * 1000)
    
    predictions = result[0][0]
    top5_idx = predictions.argsort()[-5:][::-1]
    
    print(f"ONNX Runtime inferencia OK")
    print(f"Latencia: {np.median(times):.1f} ms (mediana)")
    print(f"Top-5 índices: {top5_idx}")
    print(f"Confianza máxima: {predictions[top5_idx[0]]:.4f}")

## 6. Scaffold Automático con MLForge

MLForge genera proyectos completos y listos para compilar/ejecutar:

In [ ]:
# Generar un proyecto web de ejemplo
import sys
sys.path.insert(0, '..')

try:
    from mlforge.deploy.scaffold import scaffold, list_targets
    
    # Ver targets disponibles
    print("Targets disponibles:")
    for t in list_targets():
        print(f"  {t['name']:10s} — {t['description']} ({t['model_format']})")
    
    # Generar proyecto web
    labels = ['rosa', 'tulipán', 'girasol', 'margarita', 'orquídea']
    project_dir = scaffold(
        target='web',
        output_dir='output',
        labels=labels,
        input_size=224,
    )
    
    print(f"\nProyecto generado: {project_dir}")
    print("\nArchivos creados:")
    for f in sorted(project_dir.rglob('*')):
        if f.is_file():
            print(f"  {f.relative_to(project_dir)}")

except ImportError:
    print("Ejecuta desde el directorio del proyecto mlforge.")
    print("O usa el CLI: mlforge deploy web --labels 'rosa,tulipán,girasol'")

## Resumen: De Datos a App

```
1. mlforge train --config config.yaml           # Entrena modelo
2. mlforge export --config config.yaml -f all    # Exporta a todos los formatos
3. mlforge benchmark --model-dir exported/       # Compara rendimiento
4. mlforge deploy android -m model.tflite        # Genera app Android
5. mlforge deploy web -m model.onnx              # Genera app web
6. mlforge deploy edge -m model_edgetpu.tflite   # Genera script RPi+Coral
```

Todo el proceso: de fotos a una app funcional en cualquier plataforma.

**Siguiente notebook:** Análisis del proyecto DeepPiCar — entender el proyecto original que inspiró MLForge.